In [1]:
%cd /content

import os

# 1. 准备项目代码
REPO_DIR = "/content/bird-drone-dualstream"
REPO_URL = "https://github.com/uoi1398/bird-drone-dualstream.git"

if not os.path.exists(REPO_DIR):
    print("Repo not found. Cloning...")
    !git clone {REPO_URL}
else:
    print("Repo already exists. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull
    %cd /content

# 2. 准备公开数据集
DATASET_DIR = "/content/Drone-detection-dataset"
DATASET_URL = "https://github.com/DroneDetectionThesis/Drone-detection-dataset.git"

if not os.path.exists(DATASET_DIR):
    print("Dataset not found. Cloning...")
    !git clone {DATASET_URL}
else:
    print("Dataset already exists. Skip downloading.")

# 3. 进入项目目录
%cd {REPO_DIR}

# 4. 链接数据集
!mkdir -p data
!rm -rf data/raw
!ln -s /content/Drone-detection-dataset/Data data/raw

print("Current directory:")
!pwd

print("\nProject files:")
!ls

print("\nData raw:")
!ls data/raw

/content
Repo not found. Cloning...
Cloning into 'bird-drone-dualstream'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 65 (delta 30), reused 44 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 34.33 KiB | 11.44 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Dataset not found. Cloning...
Cloning into 'Drone-detection-dataset'...
remote: Enumerating objects: 1572, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 1572 (delta 55), reused 110 (delta 52), pack-reused 1457 (from 1)
Receiving objects: 100% (1572/1572), 291.86 MiB | 23.16 MiB/s, done.
Resolving deltas: 100% (99/99), done.
Updating files: 100% (1396/1396), done.
/content/bird-drone-dualstream
Current directory:
/content/bird-drone-dualstream

Project files:
configs  data  notebooks  README.md  requirements.txt  src

Data raw:
Audio					

In [ ]:
import sys
sys.path.append("src")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from degradation import degrade_frame
from optical_flow import frames_to_flow_rgb

In [ ]:
CSV_PATH = "data/splits/manifest.csv"
df = pd.read_csv(CSV_PATH)

row = df.iloc[0]
video_path = row["path"]

print(row)
print(video_path)

In [ ]:
def read_sample_frames(video_path, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    indices = np.linspace(0, total - 1, num_frames).astype(int)

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()

        if not ok:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

frames = read_sample_frames(video_path, num_frames=8)

print("Num frames:", len(frames))
print("Frame shape:", frames[0].shape)

In [ ]:
frame = frames[0]
degraded = degrade_frame(frame, level="strong")

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(frame)
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(degraded)
plt.title("Strong Degradation")
plt.axis("off")

plt.show()

In [ ]:
flow_frames = frames_to_flow_rgb(frames)

print("Num flow maps:", len(flow_frames))

plt.figure(figsize=(5, 5))
plt.imshow(flow_frames[0])
plt.title("Optical Flow RGB Map")
plt.axis("off")
plt.show()

In [ ]:
import os

os.makedirs("outputs/figures", exist_ok=True)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(frame)
plt.title("Original RGB")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(degraded)
plt.title("Degraded RGB")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(flow_frames[0])
plt.title("Optical Flow")
plt.axis("off")

plt.tight_layout()
plt.savefig("outputs/figures/preprocess_demo.png", dpi=300, bbox_inches="tight")
plt.show()